In [3]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt

In [2]:
def calculate_time_series_stats(file_pattern):
    """
    Reads all 30 CSV files matching a pattern, aligns them by time,
    and calculates the Mean and 95% Confidence Interval boundaries.
    """
    files = glob.glob(file_pattern)
    if not files:
        print(f"No files found for pattern: {file_pattern}")
        return None
    
    drift_runs = []
    difficulty_runs = []
    time_axis = None
    
    # Read the data metrics from each file
    for f in files:
        df = pd.read_csv(f)
        # Clean column names to strip out hidden trailing spaces
        df.columns = df.columns.str.strip()
        
        drift_runs.append(df['skew'].values)
        difficulty_runs.append(df['difficulty'].values)
        
        if time_axis is None:
            time_axis = df['t'].values / (365.25 * 86400.0) + 2009.0 # Convert seconds back to calendar years
            
    # Convert lists to 2D numpy arrays [shape: (30, number_of_printed_blocks)]
    drift_matrix = np.array(drift_runs)
    diff_matrix = np.array(difficulty_runs)
    
    N = len(files)
    
    # Calculate summary metrics across the run dimension (axis=0)
    mean_drift = np.mean(drift_matrix, axis=0)
    std_drift = np.std(drift_matrix, axis=0, ddof=1)
    ci_drift = 1.96 * (std_drift / np.sqrt(N)) # 95% Confidence Interval
    
    mean_diff = np.mean(diff_matrix, axis=0)
    std_diff = np.std(diff_matrix, axis=0, ddof=1)
    ci_diff = 1.96 * (std_diff / np.sqrt(N))
    
    return {
        'time': time_axis,
        'mean_drift': mean_drift, 'ci_drift': ci_drift,
        'mean_diff': mean_diff, 'ci_diff': ci_diff
    }

In [4]:
# 1. Process all configurations
stats_1_level = calculate_time_series_stats("data/multirun/layers_1_run_*.csv")
stats_2_level = calculate_time_series_stats("data/multirun/layers_2_run_*.csv")
stats_3_level = calculate_time_series_stats("data/multirun/layers_3_run_*.csv")
stats_4_level = calculate_time_series_stats("data/multirun/layers_4_run_*.csv")

# 2. Setup the Plotting Figure Workspace
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

# Define configurations for looping smoothly
configs = [
    (stats_1_level, '1 Level', 'blue'),
    (stats_2_level, '2 Levels', 'red'),
    (stats_3_level, '3 Levels', 'green'),
    (stats_4_level, '4 Levels', 'purple')
]

# 3. Draw the charts
for stats, label, color in configs:
    if stats is None: continue
    
    # Top Subplot: Difficulty Time-Series
    ax1.plot(stats['time'], stats['mean_diff'], label=label, color=color, linewidth=1.5)
    ax1.fill_between(stats['time'], 
                     stats['mean_diff'] - stats['ci_diff'], 
                     stats['mean_diff'] + stats['ci_diff'], 
                     color=color, alpha=0.15) # Shaded 95% CI Band

    # Bottom Subplot: Accumulated Drift Time-Series
    ax2.plot(stats['time'], stats['mean_drift'], label=label, color=color, linewidth=1.5)
    ax2.fill_between(stats['time'], 
                     stats['mean_drift'] - stats['ci_drift'], 
                     stats['mean_drift'] + stats['ci_drift'], 
                     color=color, alpha=0.15) # Shaded 95% CI Band

# Chart formatting tweaks
ax1.set_title("Difficulty Tracking with 95% Confidence Intervals (30 Runs)")
ax1.set_ylabel("Difficulty")
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

ax2.set_title("Network Drift Tracking with 95% Confidence Intervals (30 Runs)")
ax2.set_xlabel("Time (Years)")
ax2.set_ylabel("Drift (Fortnights)")
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
# plt.savefig("Robust_Statistical_Comparison.png", dpi=300)
plt.show()

/var/folders/nk/9h_rrn356s9c6szdsyk87qv80000gn/T/ipykernel_59512/3433226027.py:17: DtypeWarning: Columns (4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/core/_methods.py:269: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/core/_methods.py:258: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [ ]:
# Quick code snippet to print out text metrics for your LaTeX paper table:
for stats, label, _ in configs:
    if stats is None: continue
    final_mean = stats['mean_drift'][-1]
    final_ci = stats['ci_drift'][-1]
    print(f"{label} Final Drift Baseline: {final_mean:.4f} +/- {final_ci:.4f} Fortnights")